# Python DSA & Coding Practice

> **How to use:** read the prompt, try it yourself in a scratch cell first, *then* run the solution cell. Each solution ends with `assert` tests and prints `OK ✅` — if a cell runs clean, your logic is correct.

**Coverage:** Python fundamentals · arrays & hashing · strings · two pointers · sliding window · stack · searching/sorting · linked list · recursion/DP · Python gotchas interviewers love.

**Interview tip:** always say the **time & space complexity** out loud after you code — it's half the score. Talk through the brute force, then the optimized approach.

Related: [concepts.md](concepts.md) · [interview-qa.md](interview-qa.md) · [cheatsheet.md](cheatsheet.md)

---
## 1. Python fundamentals (warm-ups)

**Q1.1 — Reverse a string, and check if it's a palindrome.**

In [ ]:
def reverse_string(s: str) -> str:
    return s[::-1]  # slicing with step -1 — O(n) time, O(n) space

def is_palindrome(s: str) -> bool:
    return s == s[::-1]

assert reverse_string("hello") == "olleh"
assert is_palindrome("racecar") is True
assert is_palindrome("python") is False
print("OK ✅")

**Q1.2 — FizzBuzz.** Print 1..n; multiples of 3 → "Fizz", of 5 → "Buzz", of both → "FizzBuzz".

In [ ]:
def fizzbuzz(n: int) -> list[str]:
    out = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            out.append("FizzBuzz")
        elif i % 3 == 0:
            out.append("Fizz")
        elif i % 5 == 0:
            out.append("Buzz")
        else:
            out.append(str(i))
    return out

assert fizzbuzz(15)[-1] == "FizzBuzz"
assert fizzbuzz(5) == ["1", "2", "Fizz", "4", "Buzz"]
print("OK ✅")

**Q1.3 — Word frequency count.** Return a dict of word → count (case-insensitive).

In [ ]:
from collections import Counter

def word_count(text: str) -> dict[str, int]:
    return dict(Counter(text.lower().split()))

assert word_count("the cat the dog THE") == {"the": 3, "cat": 1, "dog": 1}
print("OK ✅")

**Q1.4 — Are two strings anagrams?** (same letters, any order)

In [ ]:
from collections import Counter

def is_anagram(a: str, b: str) -> bool:
    # Counter compares letter multiplicities — O(n). Sorting also works but is O(n log n).
    return Counter(a) == Counter(b)

assert is_anagram("listen", "silent") is True
assert is_anagram("hello", "world") is False
print("OK ✅")

**Q1.5 — Fibonacci** three ways: naive recursion, memoized, iterative. Know why iterative wins.

In [ ]:
from functools import lru_cache

def fib_recursive(n: int) -> int:
    # O(2^n) — exponential, recomputes the same values. Only mention as the brute force.
    if n < 2:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)

@lru_cache(maxsize=None)
def fib_memo(n: int) -> int:
    # O(n) time, O(n) space — caching turns the tree into a line.
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)

def fib_iterative(n: int) -> int:
    # O(n) time, O(1) space — the one to write in an interview.
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

assert fib_recursive(10) == fib_memo(10) == fib_iterative(10) == 55
print("OK ✅")

**Q1.6 — Swap two numbers** without a temp variable. Know the Pythonic way first; the XOR and arithmetic tricks are the "can you do it without a temp?" follow-up.

In [ ]:
# Method 1 — Pythonic tuple swap (THE interview answer): no temp variable.
a, b = 20, 30
a, b = b, a
assert (a, b) == (30, 20)

# Method 2 — classic temp variable (works in any language).
a, b = 20, 30
tmp = a
a = b
b = tmp
assert (a, b) == (30, 20)

# Method 3 — XOR swap (no temp; integers only). Clever, but rarely used in real code.
a, b = 20, 30
a = a ^ b   # a now holds 20^30
b = a ^ b   # b = (20^30)^30 = 20
a = a ^ b   # a = (20^30)^20 = 30
assert (a, b) == (30, 20)

# Method 4 — arithmetic (no temp). Fine in Python; risks overflow in fixed-width languages.
a, b = 20, 30
a = a + b   # 50
b = a - b   # 50 - 30 = 20
a = a - b   # 50 - 20 = 30
assert (a, b) == (30, 20)

print("OK ✅ every method gives a=30, b=20")

---
## 2. Arrays & hashing

> **Pattern:** a hash map (dict/set) turns an O(n²) nested-loop search into O(n) by trading space for time. This is the single most common interview trick.

**Q2.1 — Two Sum.** Return indices of the two numbers that add to `target`. (Classic hash-map problem.)

In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}  # value -> index
    for i, n in enumerate(nums):
        need = target - n
        if need in seen:          # O(1) lookup
            return [seen[need], i]
        seen[n] = i
    return []
# O(n) time, O(n) space — beats the O(n^2) brute-force double loop.

assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
print("OK ✅")

**Q2.2 — Maximum subarray sum (Kadane's algorithm).**

In [ ]:
def max_subarray(nums: list[int]) -> int:
    # At each step: extend the running sum, or start fresh at nums[i]. O(n) time, O(1) space.
    best = cur = nums[0]
    for n in nums[1:]:
        cur = max(n, cur + n)
        best = max(best, cur)
    return best

assert max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6  # [4,-1,2,1]
assert max_subarray([-1, -2, -3]) == -1
print("OK ✅")

**Q2.3 — Find the missing number** in `[0..n]` where one is missing.

In [ ]:
def missing_number(nums: list[int]) -> int:
    # Sum trick: expected total minus actual total. O(n) time, O(1) space.
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)

assert missing_number([3, 0, 1]) == 2
assert missing_number([0, 1]) == 2
print("OK ✅")

**Q2.4 — Move all zeroes to the end**, keeping the order of non-zeroes. In-place.

In [ ]:
def move_zeroes(nums: list[int]) -> list[int]:
    # Two-pointer: `pos` marks where the next non-zero goes. O(n) time, O(1) space.
    pos = 0
    for n in nums:
        if n != 0:
            nums[pos] = n
            pos += 1
    for i in range(pos, len(nums)):
        nums[i] = 0
    return nums

assert move_zeroes([0, 1, 0, 3, 12]) == [1, 3, 12, 0, 0]
print("OK ✅")

---
## 3. Strings

**Q3.1 — Reverse the words** in a sentence ("hello world" → "world hello").

In [ ]:
def reverse_words(s: str) -> str:
    return " ".join(s.split()[::-1])  # split() also collapses extra spaces

assert reverse_words("the sky is blue") == "blue is sky the"
assert reverse_words("  hello   world  ") == "world hello"
print("OK ✅")

**Q3.2 — First non-repeating character.** Return its index, or -1.

In [ ]:
from collections import Counter

def first_unique_char(s: str) -> int:
    counts = Counter(s)               # first pass: tally
    for i, ch in enumerate(s):        # second pass: first with count 1
        if counts[ch] == 1:
            return i
    return -1
# O(n) time, O(1) space (alphabet is bounded).

assert first_unique_char("leetcode") == 0
assert first_unique_char("aabb") == -1
print("OK ✅")

---
## 4. Two pointers & sliding window

> **Two pointers** = shrink/scan from both ends of sorted data. **Sliding window** = a moving range for "longest/shortest substring/subarray with property X".

**Q4.1 — Valid palindrome**, ignoring case and non-alphanumeric characters.

In [ ]:
def valid_palindrome(s: str) -> bool:
    i, j = 0, len(s) - 1
    while i < j:
        while i < j and not s[i].isalnum():
            i += 1
        while i < j and not s[j].isalnum():
            j -= 1
        if s[i].lower() != s[j].lower():
            return False
        i += 1
        j -= 1
    return True
# O(n) time, O(1) space.

assert valid_palindrome("A man, a plan, a canal: Panama") is True
assert valid_palindrome("race a car") is False
print("OK ✅")

**Q4.2 — Longest substring without repeating characters** (sliding window).

In [ ]:
def longest_unique_substring(s: str) -> int:
    last = {}          # char -> last index seen
    start = best = 0
    for i, ch in enumerate(s):
        if ch in last and last[ch] >= start:
            start = last[ch] + 1   # jump the window start past the repeat
        last[ch] = i
        best = max(best, i - start + 1)
    return best
# O(n) time, O(min(n, alphabet)) space.

assert longest_unique_substring("abcabcbb") == 3   # "abc"
assert longest_unique_substring("bbbbb") == 1
assert longest_unique_substring("pwwkew") == 3     # "wke"
print("OK ✅")

---
## 5. Stack

**Q5.1 — Valid parentheses.** Are `()[]{}` balanced and correctly nested?

In [ ]:
def valid_parentheses(s: str) -> bool:
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack   # leftover opens → invalid
# O(n) time, O(n) space.

assert valid_parentheses("()[]{}") is True
assert valid_parentheses("(]") is False
assert valid_parentheses("([{}])") is True
assert valid_parentheses("(") is False
print("OK ✅")

---
## 6. Searching & sorting

**Q6.1 — Binary search** on a sorted list. Return the index or -1. (Know it's O(log n).)

In [ ]:
def binary_search(nums: list[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
# O(log n) time, O(1) space — requires sorted input.

assert binary_search([1, 3, 5, 7, 9], 7) == 3
assert binary_search([1, 3, 5, 7, 9], 4) == -1
print("OK ✅")

**Q6.2 — Merge sort** (write at least one O(n log n) sort from scratch).

In [ ]:
def merge_sort(nums: list[int]) -> list[int]:
    if len(nums) <= 1:
        return nums
    mid = len(nums) // 2
    left = merge_sort(nums[:mid])
    right = merge_sort(nums[mid:])
    # merge two sorted halves
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:])
    out.extend(right[j:])
    return out
# O(n log n) time, O(n) space.

assert merge_sort([5, 2, 9, 1, 5, 6]) == [1, 2, 5, 5, 6, 9]
print("OK ✅")

---
## 7. Linked list

> No built-in linked list in Python, so define a `Node`. Interviewers test pointer manipulation.

In [ ]:
class Node:
    def __init__(self, val, nxt=None):
        self.val = val
        self.next = nxt

def build_list(vals):
    head = None
    for v in reversed(vals):
        head = Node(v, head)
    return head

def to_pylist(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

print("helpers ready ✅")

**Q7.1 — Reverse a linked list.**

In [ ]:
def reverse_list(head: Node) -> Node:
    prev = None
    while head:
        nxt = head.next   # save
        head.next = prev  # reverse the pointer
        prev = head       # advance prev
        head = nxt        # advance head
    return prev
# O(n) time, O(1) space.

assert to_pylist(reverse_list(build_list([1, 2, 3, 4]))) == [4, 3, 2, 1]
print("OK ✅")

**Q7.2 — Detect a cycle** (Floyd's tortoise & hare).

In [ ]:
def has_cycle(head: Node) -> bool:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next          # +1
        fast = fast.next.next     # +2 — if there's a loop, fast laps slow
        if slow is fast:
            return True
    return False
# O(n) time, O(1) space.

no_cycle = build_list([1, 2, 3])
assert has_cycle(no_cycle) is False
cyc = build_list([1, 2, 3])
cyc.next.next.next = cyc          # point tail back to head
assert has_cycle(cyc) is True
print("OK ✅")

---
## 8. Recursion & dynamic programming (basics)

**Q8.1 — Climbing stairs.** How many ways to climb `n` steps taking 1 or 2 at a time? (It's Fibonacci.)

In [ ]:
def climb_stairs(n: int) -> int:
    # ways(n) = ways(n-1) + ways(n-2). Bottom-up: O(n) time, O(1) space.
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a

assert climb_stairs(2) == 2   # (1+1), (2)
assert climb_stairs(3) == 3   # (1+1+1), (1+2), (2+1)
assert climb_stairs(5) == 8
print("OK ✅")

**Q8.2 — Coin change (min coins).** Fewest coins to make `amount`, or -1 if impossible.

In [ ]:
def coin_change(coins: list[int], amount: int) -> int:
    INF = amount + 1
    dp = [0] + [INF] * amount     # dp[x] = min coins to make x
    for x in range(1, amount + 1):
        for c in coins:
            if c <= x:
                dp[x] = min(dp[x], dp[x - c] + 1)
    return dp[amount] if dp[amount] != INF else -1
# O(amount * len(coins)) time, O(amount) space.

assert coin_change([1, 2, 5], 11) == 3   # 5+5+1
assert coin_change([2], 3) == -1
print("OK ✅")

---
## 9. Python gotchas interviewers love

> These separate "knows syntax" from "knows Python". Be ready to *explain the why*, not just fix it.

**Q9.1 — Mutable default argument.** Why is this a classic bug, and how do you fix it?

In [ ]:
# BUG: the default list is created ONCE at definition and shared across calls.
def bad_append(x, acc=[]):
    acc.append(x)
    return acc

print(bad_append(1), bad_append(2))   # prints [1, 2] [1, 2] — both names share ONE list!

# FIX: use None as the sentinel and create a fresh list inside.
def good_append(x, acc=None):
    if acc is None:
        acc = []
    acc.append(x)
    return acc

assert good_append(1) == [1]
assert good_append(2) == [2]   # independent each call
print("OK ✅")

**Q9.2 — Shallow vs deep copy.** Why does copying a nested list still share inner lists?

In [ ]:
import copy

original = [[1, 2], [3, 4]]
shallow = original.copy()          # or list(original) / original[:]
deep = copy.deepcopy(original)

shallow[0].append(99)              # mutates the SHARED inner list
assert original == [[1, 2, 99], [3, 4]]   # original changed too!

deep[0].append(77)                 # deep copy is fully independent
assert original == [[1, 2, 99], [3, 4]]   # unaffected
print("OK ✅")

**Q9.3 — Pythonic idioms** worth showing: comprehensions, `enumerate`, `zip`, `sorted(key=...)`, generators, `*args/**kwargs`.

In [ ]:
# comprehension + condition
evens = [x for x in range(10) if x % 2 == 0]
assert evens == [0, 2, 4, 6, 8]

# dict comprehension
squares = {x: x * x for x in range(4)}
assert squares == {0: 0, 1: 1, 2: 4, 3: 9}

# enumerate + zip
names, ages = ["a", "b"], [30, 40]
assert list(zip(names, ages)) == [("a", 30), ("b", 40)]

# sort by a key (e.g. by second element)
pairs = [("x", 3), ("y", 1), ("z", 2)]
assert sorted(pairs, key=lambda p: p[1]) == [("y", 1), ("z", 2), ("x", 3)]

# generator: lazy, memory-efficient (O(1) at a time vs building a list)
gen = (x * x for x in range(1_000_000))
assert next(gen) == 0 and next(gen) == 1

# *args / **kwargs
def f(*args, **kwargs):
    return sum(args) + sum(kwargs.values())
assert f(1, 2, a=3, b=4) == 10

print("OK ✅")

---
## 📝 Practice log (fill as you go)

| Problem | Solved unaided? | Complexity said out loud? | Re-drill? |
|---|---|---|---|
| Two Sum | | | |
| Kadane | | | |
| Longest unique substring | | | |
| Valid parentheses | | | |
| Reverse linked list | | | |
| Coin change | | | |

**Next steps:** re-do any "no" rows without looking; add 2–3 problems from your weakest section (usually DP or linked lists).